## PURPOSE: 
TO UNDERSTAND THE CASHFLOW ONCE MORTGAGE IS APPROVED, AND WHAT THE YEARLY POSITION MAY LOOK LIKE



**DATE:** 2026-08

**ASSUMPTIONS:**
- MORTGAGE RATE: 6.59% MAX (CAN BE LOWER)
- DAYCARE: $170/DAY NO REBATES
- EXTRA DIVISIONAL TAX TO COME OUT OF SUPERANNUATION
- EARNING POWER STAYS THE SAME
- currently only westpac and mac are considered, things that are not considered include cba, revo, and any other financial institution


In [91]:
"""Consolidate the yearly WBC CSV exports into one DataFrame."""

import pandas as pd
from pathlib import Path

DATA_DIR = Path("/Users/angie/Documents/DataFinancial")
FILE_PREFIX = "Data_export"

def load_wbc_files(data_dir: Path = DATA_DIR, prefix: str = FILE_PREFIX) -> pd.DataFrame:
    files = sorted(data_dir.glob(f"{prefix}*.csv"))
    dfs = [pd.read_csv(f, header = 0, names = ['Account','Date','Details','Debit','Credit','Balance','PaymentType','Ignore_serial']) for f in files]
    print(f"Loaded {len(dfs)} files: {[f.name for f in files]}")
    return pd.concat(dfs, ignore_index=True)

def clean_wbc_files(df: pd.DataFrame) -> pd.DataFrame:
    df['Date'] = pd.to_datetime(df['Date'])
    df['Year Month'] = df['Date'].apply(lambda x: x.strftime('%Y-%B')) 
    df['Debit'] = pd.to_numeric(df['Debit'], errors='coerce').fillna(0)
    df['Credit'] = pd.to_numeric(df['Credit'], errors='coerce').fillna(0)
    df['Total'] = (df['Credit'] - df['Debit']).abs()
    df['Bank'] = 'WBC'
    mapping = {'32134401295':'Savings_closed','732270825196':'Choice_offset', '732134697230':'Choice_offset_build', '371904307120':'Home_loan', '37190430712':'Home_loan2','6482':'Corporate_card', '8085':'Credit_card'}
    df['Account'] = df['Account'].astype(str).replace(mapping, inplace=True)
    return df

def wbc_internal_category(df_wbc: pd.DataFrame) -> pd.DataFrame:
    """Add a column for internal category based on the Details column."""
    df_wbc['Category'] = 'Default_Splurge'
    ## Income
    df_wbc.loc[df_wbc['Details'].str.contains('F072778', case=False, na=False), 'Category'] = 'Salary'

    ## Once off payments / income
    df_wbc.loc[(df_wbc['Account'] == 'Choice_offset_build') & (df_wbc['Credit'] != 0),'Category'] = 'Transfer_once_off'
    # df_wbc.loc[df_wbc['Details'].str.contains('Adsett Design Ltd|JUNAID', case=False, na=False), 'Category'] = 'Tra'

    ## Mortgage payments
    df_wbc.loc[df_wbc['Details'].str.contains('PAYMENT BY AUTHORITY TO WESTPAC BANKCORP DIREC', case=False, na=False), 'Category'] = 'MortgageCheck_WBC'
    df_wbc.loc[df_wbc['Details'].str.contains('DEPOSIT WESTPAC BANKCORPDIRECT DR 825196', case=False, na=False), 'Category'] = 'MortgageCheck_WBC'
    ## Mortgage interest
    df_wbc.loc[((df_wbc['Account'] == 'Home_loan2') & (df_wbc['Category'] == 'Default_Splurge')), 'Category'] = 'Mortgage_Interest'


    ## Credit card payments
    df_wbc.loc[df_wbc['Details'].str.contains('PAYMENT BY AUTHORITY TO CC PAYMENT', case=False, na=False), 'Category'] = 'CreditCardCheck'
    df_wbc.loc[df_wbc['Details'].str.contains('AUTOMATIC PAYMENT', case=False, na=False), 'Category'] = 'CreditCardCheck'
    # df_wbc.groupby(['Account','Details','Category'], dropna = False).agg({'Credit': 'sum','Debit':'sum'}).sort_values(by='Credit', ascending=False).head(20)
    
    ## Luxury
    df_wbc.loc[df_wbc.Details.str.contains('Hermes'),'Category'] = 'ShoppingLux'

    ## Build expenses 
    df_wbc.loc[(df_wbc['Account'] == 'Choice_offset_build') & (df_wbc['Debit'] != 0) & (df_wbc['Details'].str.contains('WITHDRAWAL')),'Category'] = 'Building_new_home'

    ## Council expenses
    df_wbc.loc[(df_wbc['Account'] == 'Choice_offset_build') & (df_wbc['Debit'] != 0) & (df_wbc['Details'].str.contains('WITHDRAWAL ONLINE')),'Category'] = 'InnerWestCouncil'

    ## high level categories
    df_wbc['HighLevelCategory'] = 'Default_Splurge_WBC'
    df_wbc.loc[df_wbc['Category'].isin(['Salary']), 'HighLevelCategory'] = 'Income'
    df_wbc.loc[df_wbc.Details.str.contains('Hermes'),'HighLevelCategory'] = 'Splurge&Firehose'
    df_wbc.loc[df_wbc['Category'].isin(['Mortgage_Interest','MortgageCheck_WBC']), 'HighLevelCategory'] = 'Mortgage'
    df_wbc.loc[df_wbc['Category'].isin(['CreditCardCheck']), 'HighLevelCategory'] = 'PayCreditCard'
    df_wbc.loc[df_wbc['Category'].isin(['InnerWestCouncil']), 'HighLevelCategory'] = 'Everyday'
    df_wbc.loc[df_wbc['Category'].isin(['Building_new_home']), 'HighLevelCategory'] = 'HomeBuild'
    df_wbc.loc[df_wbc['Category'].isin(['Transfer_once_off']), 'HighLevelCategory'] = 'Transfer'


    return df_wbc

In [92]:
## macquaire bank transactions
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/Users/angie/Documents/DataFinancial")
FILE_PREFIX = "Macquaire_"

MAC_COLUMNS = ['Date','Details','Account','Category','Subcategory','Tags','Notes','Debit','Credit','Original Description']

# The Ange 2023 export and Ben's 2021-2023 export are the same Black Card
# transactions pulled twice (935 shared rows = 90.4% of the Ange file); see the
# duplicate/overlap check cell. Drop the shared rows from the first file only —
# its remaining ~99 rows are Transaction/Savings rows that appear nowhere else.
DUPLICATE_PAIRS = [
    ("Macquaire_Ange_Transactions-2023-01-01_to_2023-12-31.csv",
     "Macquaire_Ben_Transactions-2021-01-01_to_2023-12-31-3348.csv"),
]

def _mac_row_keys(df: pd.DataFrame) -> pd.Series:
    """Fingerprint a row on the bank's own fields, ignoring editable annotations."""
    cols = [c for c in MAC_COLUMNS if c not in ('Tags', 'Notes', 'Category', 'Subcategory')]
    return df[cols].astype('string').fillna('').agg('|'.join, axis=1)


def load_macquarie_files(data_dir: Path = DATA_DIR, prefix: str = FILE_PREFIX) -> pd.DataFrame:
    files = sorted(data_dir.glob(f"{prefix}*.csv"))
    frames = {f.name: pd.read_csv(f, header = 0, names = MAC_COLUMNS) for f in files}
    print(f"Loaded {len(frames)} files: {list(frames)}")

    for drop_from, keep in DUPLICATE_PAIRS:
        missing = [n for n in (drop_from, keep) if n not in frames]
        if missing:
            # Loud, not silent: a renamed file must not quietly re-enable double-counting.
            print(f"  WARNING: no dedupe, file(s) not found (renamed?): {missing}")
            continue
        dupes = _mac_row_keys(frames[drop_from]).isin(set(_mac_row_keys(frames[keep])))
        print(f"  dropped {dupes.sum()} rows from {drop_from}"
              f" already present in {keep} ({len(frames[drop_from]) - dupes.sum()} kept)")
        frames[drop_from] = frames[drop_from][~dupes]

    return pd.concat(frames.values(), ignore_index=True)



def clean_macquarie_files(df: pd.DataFrame) -> pd.DataFrame:
    df['Date'] = pd.to_datetime(df['Date'])
    df['Year Month'] = df['Date'].apply(lambda x: x.strftime('%Y-%B')) 
    df['Debit'] = pd.to_numeric(df['Debit'], errors='coerce').fillna(0)
    df['Credit'] = pd.to_numeric(df['Credit'], errors='coerce').fillna(0)
    df['Total'] = (df['Debit'] - df['Credit']).abs()
    df['Bank'] = 'Macquarie'
    # df = df[(df['Transaction Date'] >= '2023-01-01') & (df['Transaction Date'] < '2026-01-01') & (df['Total'] != 0.0)].reset_index(drop=True)
    df = df[df['Total'] != 0.0].reset_index(drop=True)
    return df

def macquarie_internal_category(df: pd.DataFrame) -> pd.DataFrame:
    ## CLEANING CRITICAL REWRITING OF CATEGORIES, AND SUBCATEGORIES
    # INCOME
    df.loc[(df.Details.str.contains('INTEREST PAID')),'Category'] = 'Income'
    df.loc[(df.Details.str.contains('INTEREST PAID')),'Subcategory'] = 'Interest'
    df.loc[(df.Details.str.contains('Credit Kada Services|F072778')),'Category'] = 'Income'
    # DAYCARE
    # 
    df.loc[(df.Details.str.contains('Whiz Kidz')),'Category'] = 'Daycare'

    # SERVICES:
    df.loc[(df.Details.str.lower().str.contains('deft payment|service nsw')),'Category'] = 'Services'
    df.loc[(df.Details.str.lower().str.contains('city of parramatta council|service nsw')),'Category'] = 'Utilities'

    # INVESTMENT & TAX
    df.loc[(df.Details.str.lower().str.contains('building')) & (df.Category.isna()),'Category'] = 'Investment'
    df.loc[(df.Details.str.lower().str.contains('before you|swyftx|beta cash|koinly|vas|ndq|invest|mstr|biscuit')),'Category'] = 'Investment'
    df.loc[(df.Details.str.lower().str.contains('taxation')),'Category'] = 'Tax'

    # HIGH LEVEL CREDIT CARD PAYMENT
    df.loc[df['Original Description'].str.contains('BPAY PAYMENT - THANK YOU -', case = False, na = False), 'Category'] = 'CARD'
    df.loc[df['Original Description'].str.contains('BPAY MBL CARD SERVICES', case = False, na = False), 'Category'] = 'CARD'

    # MORTGAGE OFFSET SWEEP & INTEREST
    df.loc[df['Original Description'].str.contains('Repayment Sweep', case = False, na = False), 'Category'] = 'MortgageSweep'
    df.loc[(df['Account'] == 'Variable Interest') & (df['Original Description'].str.contains('Interest Charged', case = False, na = False)),'Category'] = 'InterestCharged'
    df.loc[(df['Account'] == 'Variable Interest') & (df['Original Description'].str.contains('Credit by Offset', case = False, na = False)),'Category'] = 'InterestCharged'
    df.loc[(df['Account'] == 'Variable Interest') & (df['Category']== 'Fees'),'Category'] = 'InterestCharged'

    # ONE TIME BIG TRANSFERS
    df.loc[(df['Account'] == 'Macquarie Savings Account') & (df['Original Description']== 'To Angela Chin - Funds transfer'),'Category'] = 'BIGTRANS'
    df.loc[(df['Account'] == 'Macquarie Savings Account') & (df['Original Description']== 'From KADA SERVICES PTY LTD - Directors fee'),'Category'] = 'BIGTRANS'
    
    
    df.loc[((df['Details'].str.lower().str.contains("deft")) & (df['Debit'] >= 9000)),'Category'] = 'FIX_PARK_AVE'

    #FUNDS TRANSFERS
    df.loc[df['Original Description'].str.contains('Fnds Transfer To', case = False, na = False), 'Category'] = 'FundsTransfer'
    df.loc[df['Original Description'].str.contains('From MACQM - Internal tra', case = False, na = False), 'Category'] = 'FundsTransfer'
    df.loc[df['Original Description'].str.contains('To Mortgage Account 82751921 - Internal transfer', case = False, na = False), 'Category'] = 'FundsTransfer'

    #### LUX SHOPPING
    df.loc[df.Details.str.contains('Hermes'),'Category'] = 'ShoppingLux'
    
    # GENERATE THE HIGH LEVEL CATEGORY

    df['HighLevelCategory'] = 'Splurge&Firehose' # default
    df.loc[df.Category.isin(['Income']),'HighLevelCategory'] = 'Income'
    df.loc[df.Category.isin(['Services','Health & Medical','Food & Drink','Insurance','Utilities','Transportation']),'HighLevelCategory'] = 'Everyday'
    df.loc[df.Category.isin(['Daycare']),'HighLevelCategory'] = 'Daycare'
    df.loc[df.Category.isin(['Tax']),'HighLevelCategory'] = 'Tax'
    df.loc[df.Category.isin(['CARD']),'HighLevelCategory'] = 'CHECK'
    df.loc[df.Category.isin(['MortgageSweep','InterestCharged']),'HighLevelCategory'] = 'Mortgage'
    df.loc[df.Category.isin(['Fees']),'HighLevelCategory'] = 'FinancialFees'
    df.loc[df.Category.isin(['Investment','FundsTransfer']),'HighLevelCategory'] = 'Investment'
    df.loc[df.Category.isin(['FIX_PARK_AVE']),'HighLevelCategory'] = 'Once_off'
    df.loc[df.Details.str.contains('Hermes'),'HighLevelCategory'] = 'Splurge&Firehose'

    return df



### Duplicate / overlap check across the raw exports

Run this **before** consolidating. It fingerprints every row of every export and
reports, for each pair of files, how many rows are byte-identical and what
percentage of each file that is — so a re-export that covers ground an older
file already covers shows up before it gets double-counted in `df_all`.


In [93]:
"""Check which raw export files duplicate or overlap each other, before consolidating.

Filenames lie (e.g. "..._2024-01-01_to_2023-12-31-3348.csv" actually holds 2021-2023),
so every range below is read from the data itself, never from the filename.
"""

from itertools import combinations
from pathlib import Path

import pandas as pd

DATA_DIR = Path("/Users/angie/Documents/DataFinancial")

# Annotation columns the bank lets you edit after the fact — two exports of the
# same transaction can differ here, so they are excluded from the row fingerprint.
IGNORE_COLS = {"Tags", "Notes", "Serial", "Categories", "Category", "Subcategory"}

def _row_keys(df: pd.DataFrame) -> pd.Series:
    cols = [c for c in df.columns if c not in IGNORE_COLS]
    return df[cols].astype(str).agg("|".join, axis=1)

def overlap_report(data_dir: Path = DATA_DIR, prefix: str = "", date_col: str = "Date") -> pd.DataFrame:
    """Report identical-row overlap between every pair of files matching `prefix`."""
    files = {}
    clean = 0
    print(f"--- {prefix}*.csv : files with repeated rows ---")
    for f in sorted(data_dir.glob(f"{prefix}*.csv")):
        df = pd.read_csv(f, dtype=str).fillna("")
        keys = _row_keys(df)
        dates = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)
        files[f.name] = (keys, dates)
        repeats = len(df) - keys.nunique()
        if not repeats:          # nothing repeated inside this file - not worth a line
            clean += 1
            continue
        print(f"  {f.name:58s} {len(df):5d} rows  "
              f"{dates.min():%Y-%m-%d} -> {dates.max():%Y-%m-%d}  "
              f"{repeats:3d} repeated within file")
    if clean:
        print(f"  ({clean} of {len(files)} files have no repeated rows within them)")

    rows = []
    for a, b in combinations(files, 2):
        keys_a, dates_a = files[a]
        keys_b, dates_b = files[b]
        shared = set(keys_a) & set(keys_b)
        lo, hi = max(dates_a.min(), dates_b.min()), min(dates_a.max(), dates_b.max())
        if not shared and lo > hi:
            continue  # disjoint in both rows and dates — nothing to flag
        rows.append({
            "file_a": a,
            "file_b": b,
            "shared_rows": len(shared),
            "pct_of_a": round(100 * len(shared) / max(keys_a.nunique(), 1), 1),
            "pct_of_b": round(100 * len(shared) / max(keys_b.nunique(), 1), 1),
            "date_overlap": f"{lo:%Y-%m-%d}..{hi:%Y-%m-%d}" if lo <= hi else "none",
        })
        
        
    columns = ["file_a", "file_b", "shared_rows", "pct_of_a", "pct_of_b", "date_overlap"]
    report = pd.DataFrame(rows, columns=columns).sort_values("shared_rows", ascending=False)
    # Only pairs that actually share rows get printed; pairs that merely cover the
    # same dates are still in the returned `report` for anyone who wants them.
    dupes = report[report["shared_rows"] > 0]
    print(f"\n--- {prefix}*.csv : pairs sharing identical rows ---")
    if dupes.empty:
        print(f"  no duplicate rows between any pair ({len(report)} pair(s) overlap on dates only)")
    else:
        for r in dupes.itertuples():
            print(f"  [DUPLICATE] {r.file_a}\n"
                  f"{'':14s}x {r.file_b}\n"
                  f"{'':14s}{r.shared_rows} shared rows = "
                  f"{r.pct_of_a}% of the first, {r.pct_of_b}% of the second"
                  f"   (dates {r.date_overlap})")
    return report


overlap_wbc = overlap_report(prefix="Data_export", date_col="Date")
overlap_mac = overlap_report(prefix="Macquaire_", date_col="Transaction Date")


--- Data_export*.csv : files with repeated rows ---
  Data_export_2025-01-01_2025-12-31.csv                        209 rows  2025-01-02 -> 2025-12-31    1 repeated within file
  (3 of 4 files have no repeated rows within them)

--- Data_export*.csv : pairs sharing identical rows ---
  no duplicate rows between any pair (0 pair(s) overlap on dates only)
--- Macquaire_*.csv : files with repeated rows ---
  Macquaire_Ange_Transactions-2023-01-01_to_2023-12-31.csv    1039 rows  2023-01-01 -> 2023-12-31    5 repeated within file
  Macquaire_Ben_Transactions-2019-01-01_to_2023-12-31-2210.csv  2210 rows  2019-01-02 -> 2020-12-31    9 repeated within file
  Macquaire_Ben_Transactions-2021-01-01_to_2023-12-31-3348.csv  3348 rows  2021-01-01 -> 2023-12-31   12 repeated within file
  Macquaire_Ben_Transactions-2024-01-01_to_2026-07-29-3406.csv  3406 rows  2024-01-02 -> 2026-07-28   37 repeated within file
  (3 of 7 files have no repeated rows within them)

--- Macquaire_*.csv : pairs sharing iden

In [94]:
if __name__ == "__main__":
    df_wbc = load_wbc_files()
    print(f"WBC before {len(df_wbc)} rows")
    df_wbc = clean_wbc_files(df_wbc)
    df_wbc = wbc_internal_category(df_wbc)
    print(f"WBC after {len(df_wbc)} rows")
    # print(df_wbc.head())
    
    # macquaire
    df_macquaire = load_macquarie_files()
    print("Mac before",len(df_macquaire))
    df_macquaire = clean_macquarie_files(df_macquaire)
    df_macquaire = macquarie_internal_category(df_macquaire)
    print("Mac after",len(df_macquaire))
    # print(df_macquaire.head())
    
    ## concat
    df_all = pd.concat([df_wbc, df_macquaire], ignore_index=True)
    df_all['Year'] = pd.to_datetime(df_all['Date']).dt.year
    df_all = df_all[(df_all['Date'] >= '2023-01-01') & (df_all['Date'] < '2027-01-01')].reset_index(drop=True)
    print(len(df_wbc), len(df_macquaire),len(df_all))



Loaded 4 files: ['Data_export_2023-01-1_2023-12-31.csv', 'Data_export_2024-01-01_2024-12-31.csv', 'Data_export_2025-01-01_2025-12-31.csv', 'Data_export_2026-01-01_2026-07-30.csv']
WBC before 299 rows
WBC after 299 rows
Loaded 7 files: ['Macquaire_Ange_Transactions-2023-01-01_to_2023-12-31.csv', 'Macquaire_Ange_Transactions-2024-01-01_to_2024-12-31.csv', 'Macquaire_Ange_Transactions-2025-01-01_to_2025-12-31.csv', 'Macquaire_Ange_Transactions-2026-01-01_to_2026-08-03.csv', 'Macquaire_Ben_Transactions-2019-01-01_to_2023-12-31-2210.csv', 'Macquaire_Ben_Transactions-2021-01-01_to_2023-12-31-3348.csv', 'Macquaire_Ben_Transactions-2024-01-01_to_2026-07-29-3406.csv']
  dropped 940 rows from Macquaire_Ange_Transactions-2023-01-01_to_2023-12-31.csv already present in Macquaire_Ben_Transactions-2021-01-01_to_2023-12-31-3348.csv (99 kept)
Mac before 9501
Mac after 9274
299 9274 5326


/var/folders/b3/w2v2l2_n5yn9hvnhsl1p1jwh0000gn/T/ipykernel_2039/144522402.py:16: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'])


### Flagging internal transfers

`FLAG_TRANSFER` marks both legs of a movement of our own money: a debit leaving one
account and an equal credit landing in a *different* account on the same day, where
**both** rows are worded like a transfer or a bill payment (`TRANSFER_PATTERN`).
Those rows are not real income or spend, so exclude them before summing a category.

The wording check is what keeps coincidences out - a $17.15 dinner on the same day as
a $17.15 card payment is not a transfer. Add to `TRANSFER_PATTERN` when a new wording
turns up in the exports.

`TRANSFER_PAIR_ID` then ties the two legs together so `tag_transfer_pairs` can re-label
them from what the *pair* says. That matters because only one leg usually carries the
reference text - "From Rent Savings Account" is met by a bare "To Linked Account
Xx3866 - Internal Transfer" on the other side, and a row-by-row rule would only ever
catch half of it.

The rules, in order (first match wins, both legs always get the same two labels):

| Pair wording | `HighLevelCategory` and `Category` become |
|---|---|
| contains `invest` (invest, investment, ...) | `Transfer_for_investments_ignore` |
| contains `gym` | `Transfer_between_accounts_ignore_` |
| contains `rent` (rent, rental, rents) | `Transfer_rent_between_accounts_ignore` |
| none of the above, but the pair moved >= $22,000 | `Transfer_for_investments_ignore` |

A pair is skipped entirely when **either** leg is already labelled mortgage, card or
check (`PROTECTED_CATEGORY_PATTERN`) - those are the mortgage and credit-card
reconciliation checks, and overwriting them would hide the position they exist to
show. Pairs matching no rule keep the categories they came in with.


In [95]:
"""Flag the two legs of an internal transfer between our own accounts."""

from collections import defaultdict

import numpy as np
import pandas as pd

# A row can only be a leg of a transfer if its Details read like one. Both banks
# word the same movement differently on each side, so this has to cover the
# sending wording, the receiving wording, and the free text we type ourselves.
TRANSFER_PATTERN = (
    # movement of funds
    r"transfer|\btfr\b|fnds?\s*trans|funds?\s*trans|linked\s+account|repayment\s+sweep|\bmacqm\b"
    # a bill, card or direct-debit payment
    r"|payment|\bpymt\b|\bbpay\b|\bosko\b|bankcorpdirect"
    # our own reference text on the receiving leg (edit as new wording shows up)
    r"|\binvest"
)


def _pair_across_accounts(debits, credits, account):
    """Match debit legs to credit legs in a *different* account, one-to-one.

    Kuhn's algorithm, so a debit that grabbed the only cross-account credit will
    give it up if that lets another debit match too. Each group is a handful of
    rows (same day, same cent amount), so the recursion stays shallow.
    """
    taken = {}  # credit position -> the debit position it is paired with

    def assign(d, seen):
        for c in credits:
            if account[c] == account[d] or c in seen:
                continue
            seen.add(c)
            if c not in taken or assign(taken[c], seen):
                taken[c] = d
                return True
        return False

    for d in debits:
        assign(d, set())
    return [(d, c) for c, d in taken.items()]


def flag_transfers(df_all: pd.DataFrame, decimals: int = 2,
                   pattern: str = TRANSFER_PATTERN) -> pd.DataFrame:
    """Return `df_all` with `FLAG_TRANSFER` and `TRANSFER_PAIR_ID` columns.

    `FLAG_TRANSFER` is True on both rows of every matched pair: same calendar
    day, equal amount, one row a debit and the other a credit, sitting in two
    different accounts, and *both* worded like a money transfer or a bill
    payment (`pattern`) - i.e. a movement of our own funds rather than real
    income or spend.

    `TRANSFER_PAIR_ID` carries which pair a row belongs to (-1 when unpaired),
    so downstream rules can read both legs together - one leg often carries the
    wording that says what the movement was for while the other says nothing.

    Pairing is one-to-one: four $500 debits against a single $500 credit on the
    same day flag two rows, not five.
    """
    df = df_all.copy()

    day = pd.to_datetime(df['Date']).dt.normalize().to_numpy()
    debit = pd.to_numeric(df['Debit'], errors='coerce').fillna(0).round(decimals).to_numpy()
    credit = pd.to_numeric(df['Credit'], errors='coerce').fillna(0).round(decimals).to_numpy()
    # An account name can repeat across banks ("Savings"), so key on the pair.
    account = (df['Bank'].astype(str) + '|' + df['Account'].astype(str)).to_numpy()
    looks_like_transfer = df['Details'].astype(str).str.contains(
        pattern, case=False, na=False, regex=True).to_numpy()

    # A candidate leg has exactly one side filled and reads like a transfer. The
    # wording is checked here, before pairing, so a coincidence (a $17.15 dinner
    # landing on the same day as a $17.15 card payment) never consumes a leg that
    # a real transfer needed.
    is_debit = (debit > 0) & (credit == 0) & looks_like_transfer
    is_credit = (credit > 0) & (debit == 0) & looks_like_transfer
    amount = np.where(is_debit, debit, credit)

    # Bucket the candidates by (day, amount): each bucket then holds every row
    # that could possibly be the counterpart of another row in it.
    buckets = defaultdict(lambda: ([], []))
    for i in np.flatnonzero(is_debit | is_credit):
        buckets[(day[i], amount[i])][0 if is_debit[i] else 1].append(i)

    flag = np.zeros(len(df), dtype=bool)
    pair_id = np.full(len(df), -1, dtype=np.int64)
    next_id = 0
    for debits, credits in buckets.values():
        if not debits or not credits:
            continue
        for d, c in _pair_across_accounts(debits, credits, account):
            flag[d] = flag[c] = True
            pair_id[d] = pair_id[c] = next_id
            next_id += 1

    df['FLAG_TRANSFER'] = flag
    df['TRANSFER_PAIR_ID'] = pair_id
    return df


# --- Re-labelling a flagged pair from its own wording -----------------------

# Checked against `HighLevelCategory` and `Category` on *either* leg. These
# labels are the mortgage and credit-card reconciliation checks - the pair is
# already saying something we need, so the rules below leave it alone rather
# than overwrite it. Covers Mortgage / MortgageCheck / MortgageSweep / CARD /
# CHECK / CreditCardCheck.
PROTECTED_CATEGORY_PATTERN = r"mortgage|card|check"

# Tried in order against the pooled wording of both legs, first match wins. One
# leg usually carries the reference text ("- Gym", "From Rent Savings Account")
# and the other is silent, which is why this reads the pair and not the row.
PAIR_RULES = [
    (r"\binvest", 'Transfer_for_investments_ignore'),
    (r"\bgym\b", 'Transfer_between_accounts_ignore_'),
    (r"\brent(?:al|als|s)?\b", 'Transfer_rent_between_accounts_ignore'),
]

# Fallback for the big unworded movements (the 2023 build drawdowns, the
# deposit): a same-day matched pair this size is us moving capital, not spend.
LARGE_TRANSFER_THRESHOLD = 22000
LARGE_TRANSFER_LABEL = 'Transfer_for_investments_ignore'


def tag_transfer_pairs(df_all: pd.DataFrame,
                       rules=PAIR_RULES,
                       protected: str = PROTECTED_CATEGORY_PATTERN,
                       threshold: float = LARGE_TRANSFER_THRESHOLD,
                       threshold_label: str = LARGE_TRANSFER_LABEL) -> pd.DataFrame:
    """Set `HighLevelCategory` and `Category` on both legs of a flagged pair.

    A pair is skipped entirely when either leg is labelled mortgage, card or
    check (`protected`). Otherwise the first matching rule in `rules` wins, and
    a pair matching none of them still gets `threshold_label` if it moved at
    least `threshold`. Both legs always end up with the same two labels.
    """
    df = df_all.copy()
    paired = df['TRANSFER_PAIR_ID'] >= 0
    if not paired.any():
        return df

    description = df['Details'].astype(str)
    if 'Original Description' in df.columns:
        # Macquarie keeps our reference text here and truncates it in Details.
        description = description + ' ' + df['Original Description'].fillna('').astype(str)
    labels = df['HighLevelCategory'].astype(str) + ' ' + df['Category'].astype(str)
    amount = pd.to_numeric(df['Total'], errors='coerce').fillna(0)

    legs = pd.DataFrame({
        'pair': df.loc[paired, 'TRANSFER_PAIR_ID'],
        'text': description[paired].str.lower(),
        'protected': labels[paired].str.contains(protected, case=False, na=False),
        'amount': amount[paired],
    })
    by_pair = legs.groupby('pair')
    pair_text = by_pair['text'].agg(' | '.join)
    open_pair = ~by_pair['protected'].any()          # mortgage/card/check legs opt the pair out
    pair_amount = by_pair['amount'].max()

    tag = pd.Series(pd.NA, index=pair_text.index, dtype='object')
    for pattern, label in rules:
        tag[tag.isna() & open_pair & pair_text.str.contains(pattern, regex=True)] = label
    tag[tag.isna() & open_pair & (pair_amount >= threshold)] = threshold_label

    retag = df['TRANSFER_PAIR_ID'].map(tag)
    hit = retag.notna()
    df.loc[hit, 'HighLevelCategory'] = retag[hit]
    df.loc[hit, 'Category'] = retag[hit]
    print(f"Re-tagged {hit.sum()} rows across {tag.notna().sum()} transfer pairs:")
    print(retag[hit].value_counts().to_string())
    return df


df_all = flag_transfers(df_all)
print(f"FLAG_TRANSFER: {df_all['FLAG_TRANSFER'].sum()} of {len(df_all)} rows "
      f"(${df_all.loc[df_all['FLAG_TRANSFER'], 'Total'].sum():,.2f} of movement)")

df_all = tag_transfer_pairs(df_all)
df_all.loc[(df_all['FLAG_TRANSFER'] == True) & (df_all['Category'] == 'MortgageCheck'), 'HighLevelCategory'] = 'MortgageCheck'

df_all[df_all['FLAG_TRANSFER']].sort_values('Total', ascending=False).to_csv(DATA_DIR / "flagged_transfers_df_all.csv", index=False)


FLAG_TRANSFER: 482 of 5326 rows ($1,374,566.64 of movement)
Re-tagged 160 rows across 80 transfer pairs:
TRANSFER_PAIR_ID
Transfer_for_investments_ignore          116
Transfer_rent_between_accounts_ignore     28
Transfer_between_accounts_ignore_         16


## Spliting costs by year

In [96]:
## split by each calendar year
def each_year(df: pd.DataFrame, year = 2026) -> pd.DataFrame:
    return df[df['Year Month'].str.contains(str(year))].reset_index(drop=True).sort_values(by='Date', ascending=True)

df_2026 = each_year(df_all, 2026)
df_2025 = each_year(df_all, 2025)
df_2024 = each_year(df_all, 2024)
df_2023 = each_year(df_all, 2023)

print(len(df_2026), len(df_2025), len(df_2024), len(df_2023))

1024 1885 1152 1265


In [97]:
df_2023 = each_year(df_all, 2023)
df_2023_summary = df_2023.groupby(['HighLevelCategory','Category','FLAG_TRANSFER'],dropna= False).agg({'Total': 'sum','Credit': 'sum','Debit': 'sum'}).reset_index().sort_values(by=['HighLevelCategory','Total'], ascending=False).reset_index(drop=True)
df_2023_summary.columns = ['HighLevelCategory','Category','FLAG_TRANSFER','Total2023','Credit2023','Debit2023']
df_2024 = each_year(df_all, 2024)
df_2024_summary = df_2024.groupby(['HighLevelCategory','Category','FLAG_TRANSFER'],dropna= False).agg({'Total': 'sum','Credit': 'sum','Debit': 'sum'}).reset_index().sort_values(by=['HighLevelCategory','Total'], ascending=False).reset_index(drop=True)
df_2024_summary.columns = ['HighLevelCategory','Category','FLAG_TRANSFER','Total2024','Credit2024','Debit2024']
df_2025 = each_year(df_all, 2025)
df_2025_summary = df_2025.groupby(['HighLevelCategory','Category','FLAG_TRANSFER'],dropna= False).agg({'Total': 'sum','Credit': 'sum','Debit': 'sum'}).reset_index().sort_values(by=['HighLevelCategory','Total'], ascending=False).reset_index(drop=True)
df_2025_summary.columns = ['HighLevelCategory','Category','FLAG_TRANSFER','Total2025','Credit2025','Debit2025']
df_2026 = each_year(df_all, 2026)
df_2026_summary = df_2026.groupby(['HighLevelCategory','Category','FLAG_TRANSFER'],dropna= False).agg({'Total': 'sum','Credit': 'sum','Debit': 'sum'}).reset_index().sort_values(by=['HighLevelCategory','Total'], ascending=False).reset_index(drop=True)
df_2026_summary.columns = ['HighLevelCategory','Category','FLAG_TRANSFER','Total2026','Credit2026','Debit2026']

df_all_summary_category = df_2026_summary.merge(df_2025_summary, on=['HighLevelCategory','Category','FLAG_TRANSFER'], how='outer').fillna(0).sort_values(by=['HighLevelCategory','Total2026'], ascending=False).reset_index(drop=True)
df_all_summary_category = df_all_summary_category.merge(df_2024_summary, on=['HighLevelCategory','Category','FLAG_TRANSFER'], how='outer').fillna(0).sort_values(by=['HighLevelCategory','Total2026'], ascending=False).reset_index(drop=True)
df_all_summary_category = df_all_summary_category.merge(df_2023_summary, on=['HighLevelCategory','Category','FLAG_TRANSFER'], how='outer').fillna(0).sort_values(by=['HighLevelCategory','Total2026'], ascending=False).reset_index(drop=True)

df_all.sort_values(by='Date',ascending = False).to_csv(DATA_DIR/'df_all_MASTER.csv',index=False)

df_all[(df_all['HighLevelCategory'].isin(['Splurge&Firehose','Default_Splurge_WBC'])) & ~(df_all.Category.isin(['Financial','Uncategorized']))].sort_values(by='Date',ascending = False).to_csv(DATA_DIR/'df_all_Splurge&Firehose_nonFI_nonuncat.csv',index=False)

df_all_summary_category 

,HighLevelCategory,Category,FLAG_TRANSFER,Total2026,Credit2026,Debit2026,Total2025,Credit2025,Debit2025,Total2024,Credit2024,Debit2024,Total2023,Credit2023,Debit2023
0,Transfer_rent_between_accounts_ignore,Transfer_rent_between_accounts_ignore,True,48727.86,24363.93,24363.93,48443.60,24221.80,24221.80,0.00,0.00,0.00,0.00,0.00,0.00
1,Transfer_for_investments_ignore,Transfer_for_investments_ignore,True,120000.00,60000.00,60000.00,276000.00,138000.00,138000.00,140000.00,70000.00,70000.00,388980.00,194490.00,194490.00
2,Transfer_between_accounts_ignore_,Transfer_between_accounts_ignore_,True,0.00,0.00,0.00,600.00,300.00,300.00,1000.00,500.00,500.00,0.00,0.00,0.00
3,Tax,Tax,False,25980.09,25980.09,0.00,1872.49,0.64,1871.85,6423.35,0.00,6423.35,8444.55,0.00,8444.55
4,Splurge&Firehose,Financial,False,67996.50,26030.50,41966.00,88522.54,51092.27,37430.27,201635.64,134431.06,67204.58,1804698.73,959930.56,844768.17
5,Splurge&Firehose,BIGTRANS,False,60900.00,60900.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
6,Splurge&Firehose,Financial,True,34129.50,18090.13,16039.37,37887.26,16168.63,21718.63,23356.02,11678.01,11678.01,15055.70,7527.85,7527.85
7,Splurge&Firehose,Travel,False,6383.67,0.00,6383.67,4209.07,262.51,3946.56,6764.69,1182.01,5582.68,8721.91,0.00,8721.91
8,Splurge&Firehose,ShoppingLux,False,6080.00,0.00,6080.00,830.00,0.00,830.00,370.00,0.00,370.00,320.00,0.00,320.00
9,Splurge&Firehose,Personal,False,4084.88,19.96,4064.92,5590.94,0.00,5590.94,4389.08,55.09,4333.99,6920.92,133.64,6787.28


### Plots per category

In [98]:
"""Monthly category detail — one small panel per category, one colour.

Readability picked the form here. A stacked bar with eight categories asks the eye
to tell eight muted pinks apart, and in this palette that is not possible: the
worst pair measures ΔE 5.5 against a floor of 15, which is exactly why the stacks
were unreadable. Giving every category its own panel and its own baseline removes
the problem — nothing is identified by colour, so a single rose does the whole
chart and the greys are free to stay chrome.
"""

import math

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --------------------------------------------------------------------- palette
SURFACE   = "#FCFCFB"   # chart + paper ground
INK       = "#2B2724"   # headings, panel names
INK_SOFT  = "#6B6560"   # subtitles, annotations
INK_MUTED = "#9A938E"   # tick labels, secondary numbers
GRID      = "#EDE9E7"   # hairline grid
RULE      = "#DBD5D2"   # baseline / axis

ROSE      = "#7E4B5B"                  # the one data colour
ROSE_WASH = "rgba(126, 75, 91, 0.13)"
GREY      = "#B7B1AB"                  # the second mark, where there are two
GREY_WASH = "rgba(183, 177, 171, 0.22)"
PLUM      = "#3A2A30"                  # net line / emphasis

FONT = "Inter, Avenir Next, -apple-system, Helvetica Neue, Arial, sans-serif"


def pretty(name) -> str:
    """Panel and legend text is prose, not a column name."""
    return str(name).replace("_", " ").replace("&", " & ").replace("  ", " ").strip()


def monthly_pivot(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df["Year Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
    return (
        df.groupby(["Year Month", "Category"])[value_col]
        .sum()
        .reset_index()
        .pivot(index="Year Month", columns="Category", values=value_col)
        .fillna(0)
        .sort_index()
    )


def build_small_multiples(df, value_col, title, subtitle="", panels=8, cols=3,
                          shared_scale=False):
    """A panel per category, biggest first, each panel scaled to its own range.

    Own-scale is the default because a shared one flattens every small category
    into the baseline — the panel's own axis and the total in its heading carry
    the size instead. `shared_scale=True` makes the panels directly comparable at
    the cost of that detail.
    """
    pivot = monthly_pivot(df, value_col)
    totals = pivot.sum().sort_values(ascending=False)
    kept = list(totals.index[:panels])
    tail = [c for c in pivot.columns if c not in kept]

    grid = pivot.reindex(columns=kept)
    if tail:
        grid["Other"] = pivot[tail].sum(axis=1)

    names = list(grid.columns)
    rows = math.ceil(len(names) / cols)
    ceiling = float(grid.to_numpy().max()) or 1.0

    fig = make_subplots(
        rows=rows, cols=cols,
        shared_xaxes=True, shared_yaxes=shared_scale,
        vertical_spacing=0.13, horizontal_spacing=0.075,
        subplot_titles=[
            f"<b>{pretty(n)}</b>  <span style='color:{INK_MUTED}'>${grid[n].sum():,.0f}</span>"
            for n in names
        ],
    )

    for i, name in enumerate(names):
        row, col = divmod(i, cols)
        series = grid[name]
        bucket = name == "Other"          # the fold-in reads grey, not as a category
        stroke, wash = (GREY, GREY_WASH) if bucket else (ROSE, ROSE_WASH)

        fig.add_trace(go.Scatter(
            x=series.index, y=series.values,
            mode="lines",
            line=dict(color=stroke, width=2, shape="spline", smoothing=0.35),
            fill="tozeroy", fillcolor=wash,
            hovertemplate=(
                f"<b>{pretty(name)}</b><br>%{{x|%b %Y}}<br>$%{{y:,.0f}}<extra></extra>"
            ),
            showlegend=False,
        ), row=row + 1, col=col + 1)

        fig.add_trace(go.Scatter(      # where it sits now, ringed in the surface colour
            x=[series.index[-1]], y=[series.values[-1]],
            mode="markers",
            marker=dict(size=8, color=stroke, line=dict(color=SURFACE, width=2)),
            hoverinfo="skip", showlegend=False,
        ), row=row + 1, col=col + 1)

    fig.update_xaxes(
        showgrid=False,
        showline=True, linecolor=RULE, linewidth=1,
        ticks="outside", tickcolor=RULE, ticklen=3,
        tickfont=dict(size=10, color=INK_MUTED),
        tickformat="%Y", dtick="M12",
        range=[grid.index.min() - pd.Timedelta(days=30),
               grid.index.max() + pd.Timedelta(days=40)],   # room for the end dot
    )
    fig.update_yaxes(
        showgrid=True, gridcolor=GRID, gridwidth=1,
        zeroline=False, showline=False, ticks="",
        tickfont=dict(size=10, color=INK_MUTED),
        tickprefix="$", tickformat="~s", nticks=3,
        **({"range": [0, ceiling * 1.16]} if shared_scale
           else {"rangemode": "tozero", "showticklabels": True}),
    )

    for empty in range(len(names), rows * cols):     # no ghost axes in a part-full row
        row, col = divmod(empty, cols)
        fig.update_xaxes(visible=False, row=row + 1, col=col + 1)
        fig.update_yaxes(visible=False, row=row + 1, col=col + 1)

    fig.update_layout(
        template="none",
        font=dict(family=FONT, size=11, color=INK_SOFT),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
        height=126 + rows * 168,
        margin=dict(l=66, r=34, t=118, b=54),
        showlegend=False,
        title=dict(
            text=f"<b>{title}</b>",
            subtitle=dict(text=subtitle, font=dict(size=13, color=INK_MUTED)),
            font=dict(size=21, color=INK),
            x=0, xref="paper", xanchor="left", y=0.97, yanchor="top",
        ),
        hoverlabel=dict(
            bgcolor=SURFACE, bordercolor=GRID,
            font=dict(family=FONT, size=12, color=INK), align="left",
        ),
    )

    for i, note in enumerate(fig.layout.annotations[:len(names)]):
        axis = "xaxis" if i == 0 else f"xaxis{i + 1}"     # sit each title on its own panel
        note.update(
            x=fig.layout[axis].domain[0], xanchor="left",
            font=dict(size=12, color=INK), yshift=5,
        )

    return fig


scale_note = "each panel on its own scale — the heading carries the total"

spend_fig = build_small_multiples(
    df_all, "Debit",
    "Where the money goes",
    f"Monthly spend by category · biggest first · {scale_note}",
)
income_fig = build_small_multiples(
    df_all, "Credit",
    "Where the money comes from",
    f"Monthly credits by category · biggest first · {scale_note}",
)

spend_fig.show()
income_fig.show()

/var/folders/b3/w2v2l2_n5yn9hvnhsl1p1jwh0000gn/T/ipykernel_2039/1832652036.py:115: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  range=[grid.index.min() - pd.Timedelta(days=30),
/var/folders/b3/w2v2l2_n5yn9hvnhsl1p1jwh0000gn/T/ipykernel_2039/1832652036.py:116: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  grid.index.max() + pd.Timedelta(days=40)],   # room for the end dot
/var/folders/b3/w2v2l2_n5yn9hvnhsl1p1jwh0000gn/T/ipykernel_2039/1832652036.py:115: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit 

In [99]:
"""Money in against money out — the bottom line, with category detail left to the panels above.

Two marks only, and they never touch: credits above the baseline, debits below.
Direction is the primary encoding, which leaves colour redundant — and that is what
keeps a near-neutral rose-and-grey pair legible where eight of them were not.
Validated as a pair: ΔE 28.5 under deuteranopia, 29.5 normal vision.
"""

import math


def _nice_step(span: float, target_ticks: int = 4) -> float:
    """Round tick spacing up to a 1 / 2 / 2.5 / 5 × 10ⁿ step."""
    if span <= 0:
        return 1.0
    raw = span / max(target_ticks, 1)
    mag = 10 ** math.floor(math.log10(raw))
    for m in (1, 2, 2.5, 5):
        if raw <= m * mag:
            return m * mag
    return 10 * mag


def _symmetric_axis(up: float, down: float, headroom: float = 1.12):
    """Mirrored around zero, labelled as magnitudes — no '−$' below the line.

    Ticks are clipped to the data's own reach, so the axis never runs on past the
    tallest bar just to land on a round number.
    """
    reach = max(up, down)
    limit = reach * headroom
    step = _nice_step(reach)
    n = int(limit // step)
    values = [step * i for i in range(-n, n + 1)]
    return values, [f"${abs(v):,.0f}" for v in values], [-limit, limit]


source = df_wbc
flow = source.copy()
flow["Date"] = pd.to_datetime(flow["Date"])
flow["Year Month"] = flow["Date"].dt.to_period("M").dt.to_timestamp()
flow = flow.groupby("Year Month")[["Credit", "Debit"]].sum().sort_index()
net = flow["Credit"] - flow["Debit"]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=flow.index, y=flow["Credit"], name="In",
    marker=dict(color=GREY, line=dict(color=SURFACE, width=1)),
    hovertemplate="$%{y:,.0f}<extra>In</extra>",
))
fig.add_trace(go.Bar(
    x=flow.index, y=-flow["Debit"], name="Out",
    marker=dict(color=ROSE, line=dict(color=SURFACE, width=1)),
    customdata=flow["Debit"].to_numpy(),
    hovertemplate="$%{customdata:,.0f}<extra>Out</extra>",
))
fig.add_trace(go.Scatter(
    x=net.index, y=net.values, name="Net",
    mode="lines+markers",
    line=dict(color=PLUM, width=2, shape="spline", smoothing=0.4),
    marker=dict(size=7, color=PLUM, line=dict(color=SURFACE, width=2)),
    hovertemplate="$%{y:,.0f}<extra>Net</extra>",
))

ticks, labels, span = _symmetric_axis(flow["Credit"].max(), flow["Debit"].max())

fig.update_layout(
    template="none",
    barmode="relative",
    bargap=0.4,
    barcornerradius=2,
    font=dict(family=FONT, size=12, color=INK_SOFT),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    height=580,
    margin=dict(l=78, r=44, t=112, b=88),
    hovermode="x unified",
    title=dict(
        text="<b>Money in vs money out</b>",
        subtitle=dict(
            text="Westpac · credits above the line, debits below · the plum line is the monthly net",
            font=dict(size=13, color=INK_MUTED),
        ),
        font=dict(size=21, color=INK),
        x=0, xref="paper", xanchor="left", y=0.95, yanchor="top",
    ),
    legend=dict(
        orientation="h", traceorder="normal",
        x=0, xanchor="left", y=-0.16, yanchor="top",
        font=dict(size=12, color=INK_SOFT),
        itemsizing="constant", itemwidth=30,
        bgcolor="rgba(0,0,0,0)", title_text="",
    ),
    hoverlabel=dict(
        bgcolor=SURFACE, bordercolor=GRID,
        font=dict(family=FONT, size=12, color=INK), align="left",
    ),
    xaxis=dict(
        title_text="",
        showgrid=False,
        showline=True, linecolor=RULE, linewidth=1,
        ticks="outside", tickcolor=RULE, ticklen=4,
        tickfont=dict(size=11, color=INK_MUTED),
        tickformat="%b<br>%Y", dtick="M3",
        range=[flow.index.min() - pd.Timedelta(days=18),
               flow.index.max() + pd.Timedelta(days=18)],
    ),
    yaxis=dict(
        title_text="",
        showgrid=True, gridcolor=GRID, gridwidth=1,
        zeroline=True, zerolinecolor=RULE, zerolinewidth=1,
        showline=False, ticks="",
        tickfont=dict(size=11, color=INK_MUTED),
        tickmode="array", tickvals=ticks, ticktext=labels, range=span,
    ),
)

# Direction said in words, so the reader never has to decode it from the two fills.
for text, y, anchor in (("money in ↑", span[1], "top"), ("money out ↓", span[0], "bottom")):
    fig.add_annotation(
        x=1, xref="paper", xanchor="right", y=y, yanchor=anchor,
        text=text, showarrow=False, font=dict(size=11, color=INK_MUTED),
    )

# Two direct labels, not a number on every bar: the worst month and the latest one.
if len(net) and net.min() < 0:
    worst = net.idxmin()
    position = net.index.get_loc(worst) / max(len(net) - 1, 1)
    fig.add_annotation(
        x=worst, y=-flow["Debit"].loc[worst],
        xanchor="right" if position > 0.85 else "left" if position < 0.15 else "center",
        text=f"worst month · {worst:%b %Y} · −${abs(net.min()):,.0f}",
        showarrow=False, yshift=-12, yanchor="top",
        font=dict(size=11, color=INK_SOFT),
    )

if len(net):
    latest = net.index[-1]
    fig.add_annotation(
        x=latest, y=net.iloc[-1],
        text=f"  net {net.iloc[-1]:+,.0f}".replace("+", "+$").replace("-", "−$"),
        showarrow=False, xanchor="left", yshift=2,
        font=dict(size=11, color=PLUM),
    )

fig.show()

/var/folders/b3/w2v2l2_n5yn9hvnhsl1p1jwh0000gn/T/ipykernel_2039/2522642072.py:104: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  range=[flow.index.min() - pd.Timedelta(days=18),
/var/folders/b3/w2v2l2_n5yn9hvnhsl1p1jwh0000gn/T/ipykernel_2039/2522642072.py:105: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  flow.index.max() + pd.Timedelta(days=18)],


# HIGH LEVEL EXPENSE PROJECTIONS

### Mortgage Repayment
Below is the calculation for loan repayments


In [100]:
## Mortgage
# summary $2273090
def monthly_repayment(principal, annual_rate, term_months):
    """
    principal    : amount borrowed (e.g. 300000)
    annual_rate  : annual interest rate as a percent (e.g. 6.5 for 6.5%)
    term_months  : number of monthly payments (e.g. 360 for 30 years)
    """
    monthly_rate = annual_rate / 100 / 12

    if monthly_rate == 0:                      # zero-interest edge case
        return principal / term_months

    factor = (1 + monthly_rate) ** term_months
    payment = principal * (monthly_rate * factor) / (factor - 1)
    return payment


# Example
for i in [6.09, 6.59, 6.99, 7.09, 7.19, 7.29, 7.39, 7.49, 7.59]:
    p = monthly_repayment(2273090, i, 30*12)
    print(f'Monthly repayment: ${p:,.2f}, for interest rate {i}% over 30 years on principal')
    print(f'Fortnightly repayment: ${p:,.2f}, for interest rate {i}% over 30 years on principal')



Monthly repayment: $13,760.13, for interest rate 6.09% over 30 years on principal
Fortnightly repayment: $13,760.13, for interest rate 6.09% over 30 years on principal
Monthly repayment: $14,502.28, for interest rate 6.59% over 30 years on principal
Fortnightly repayment: $14,502.28, for interest rate 6.59% over 30 years on principal
Monthly repayment: $15,107.66, for interest rate 6.99% over 30 years on principal
Fortnightly repayment: $15,107.66, for interest rate 6.99% over 30 years on principal
Monthly repayment: $15,260.57, for interest rate 7.09% over 30 years on principal
Fortnightly repayment: $15,260.57, for interest rate 7.09% over 30 years on principal
Monthly repayment: $15,414.08, for interest rate 7.19% over 30 years on principal
Fortnightly repayment: $15,414.08, for interest rate 7.19% over 30 years on principal
Monthly repayment: $15,568.20, for interest rate 7.29% over 30 years on principal
Fortnightly repayment: $15,568.20, for interest rate 7.29% over 30 years on pr

Paying earlier by paying every 2 weeks, rather than 2 payments a month. 

In [101]:
from datetime import date, timedelta
import calendar

def add_months(d, k):
    m = d.month - 1 + k
    y = d.year + m // 12
    m = m % 12 + 1
    day = min(d.day, calendar.monthrange(y, m)[1])
    return date(y, m, day)

def simulate(principal, annual_rate, payment, start, mode, days_per_year=365, max_years=60):
    """Day-by-day daily accrual. mode='monthly' or 'fortnight'."""
    daily_rate = annual_rate / 100 / days_per_year
    balance, accrued, total_interest, n = principal, 0.0, 0.0, 0
    d = start
    next_pay = add_months(start, 1) if mode == 'monthly' else start + timedelta(days=14)
    end_limit = start + timedelta(days=int(max_years * 365.25))

    while balance > 0.005 and d < end_limit:
        accrued += balance * daily_rate          # accrue one day's interest
        d += timedelta(days=1)
        if d >= next_pay:                        # payment day: capitalise + pay
            balance += accrued
            total_interest += accrued
            accrued = 0.0
            pay = min(payment, balance)
            balance -= pay
            n += 1
            next_pay = add_months(next_pay, 1) if mode == 'monthly' else next_pay + timedelta(days=14)

    return (d - start).days / 365.25, total_interest, n

def monthly_payment(principal, annual_rate, term_years, days_per_year=365):
    r = (1 + annual_rate/100/days_per_year) ** (days_per_year/12) - 1
    n = term_years * 12
    f = (1 + r) ** n
    return principal * (r * f) / (f - 1)


def compare(principal, annual_rate, term_years, start=date(2025, 1, 1)):
    m_pmt = monthly_payment(principal, annual_rate, term_years)
    f_pmt = m_pmt / 2                                   # half the monthly, paid fortnightly

    m_term, m_int, m_n = simulate(principal, annual_rate, m_pmt, start, 'monthly')
    f_term, f_int, f_n = simulate(principal, annual_rate, f_pmt, start, 'fortnight')

    print(f"Loan ${principal:,.0f} @ {annual_rate}% over {term_years} yrs (daily accrual)\n")
    print(f"1) Monthly")
    print(f"   payment:        ${m_pmt:,.2f}")
    print(f"   term:           {m_term:.2f} yrs ({m_n} payments)")
    print(f"   total interest: ${m_int:,.2f}\n")
    print(f"3) Fortnightly (half the monthly)")
    print(f"   payment:        ${f_pmt:,.2f}")
    print(f"   term:           {f_term:.2f} yrs ({f_n} payments)")
    print(f"   total interest: ${f_int:,.2f}\n")
    print(f"   -> {m_term - f_term:.2f} yrs sooner, ${m_int - f_int:,.2f} interest saved")
    return m_pmt


# Example — replace with your own numbers
INTEREST_RATE = 6.09+.25+.25
monthly_mortgage = compare(2273090, INTEREST_RATE, 30)

Loan $2,273,090 @ 6.59% over 30 yrs (daily accrual)

1) Monthly
   payment:        $14,528.60
   term:           29.91 yrs (359 payments)
   total interest: $2,932,056.47

3) Fortnightly (half the monthly)
   payment:        $7,264.30
   term:           23.76 yrs (620 payments)
   total interest: $2,230,645.93

   -> 6.15 yrs sooner, $701,410.54 interest saved


## Everyday



### Daycare costs (ACTUALS)

Below are the daycare costs for the last 3 years, with 2026 being incomplete

In [102]:
daycare_summary = df_all[df_all.Category=='Daycare'].groupby(['Year','HighLevelCategory','Category'], dropna = False).agg({'Credit': 'sum','Debit':'sum', 'Date':'count'}).sort_values(by='Credit', ascending=False)
max_daycare_cost = daycare_summary['Debit'].max()
mean_daycare_cost = daycare_summary['Debit'].mean()
print(f"Average daycare cost per year: ${mean_daycare_cost:,.2f}")
print(f"Maximum daycare cost per year: ${max_daycare_cost:,.2f}")
daycare_summary



Average daycare cost per year: $10,839.68
Maximum daycare cost per year: $13,444.70


,,,Credit,Debit,Date
Year,HighLevelCategory,Category,,,
2023,Daycare,Daycare,0.0,9842.49,23
2024,Daycare,Daycare,0.0,11066.59,26
2025,Daycare,Daycare,0.0,13444.70,26
2026,Daycare,Daycare,0.0,9004.95,15


### Daycare (assuming no rebates)

In [103]:
Daily_rate = 170
days_per_week = 4

day_care_annual_cost = Daily_rate * days_per_week * 52
day_care_monthly_cost = day_care_annual_cost / 12

print(f"DayCare Monthly cost: ${day_care_monthly_cost:,.2f}")
print(f"DayCare Annual cost: ${day_care_annual_cost:,.2f}")


DayCare Monthly cost: $2,946.67
DayCare Annual cost: $35,360.00


### Everyday cost - (ACTUALS)

In [104]:
df_all[df_all['HighLevelCategory'] == 'Everyday'].groupby('Year').agg({'Total': 'sum'}).reset_index().sort_values(by='Year', ascending=False)

,Year,Total
3,2026,23022.55
2,2025,38469.94
1,2024,32937.89
0,2023,36975.23


Side note: What happened in 2023 where the expenses were so high? Is it because we're missing expense in 2024 and 2025 or is there an outlier in 2023? Ended up trouble shooting and there was a duplicates transactions in two mac files, duplicate function implemented now


### Splurge

In [105]:

df_all[(df_all['HighLevelCategory'] == 'Splurge&Firehose') & ~(df_all.Category.isin(['Financial','Uncategorized']))].groupby(['Year','FLAG_TRANSFER'],dropna= False).agg({'Total': 'sum','Credit':'sum','Debit':'sum'}).reset_index().sort_values(by='Year', ascending=False)



,Year,FLAG_TRANSFER,Total,Credit,Debit
3,2026,False,80021.03,61506.81,18514.22
4,2026,True,2050.76,0.00,2050.76
2,2025,False,14272.58,274.90,13997.68
1,2024,False,221510.73,72582.53,148928.20
0,2023,False,115061.83,96242.82,18819.01


In [106]:
everyday_annual_summary = df_all[df_all['HighLevelCategory'] == 'Everyday'].groupby('Year')['Total'].sum()
print(f"Annual everyday expenses: {everyday_annual_summary}")
everyday_annual = max(everyday_annual_summary)
print(f"Maximum annual everyday expenses: {everyday_annual}")

Annual everyday expenses: Year
2023    36975.23
2024    32937.89
2025    38469.94
2026    23022.55
Name: Total, dtype: float64
Maximum annual everyday expenses: 38469.94


# HIGH LEVEL INCOME PROJECTIONS


COMBINED INCOME

In [107]:
b_monthly = 9176.00
a_monthly = 6800.73 * 2
## numbers taken from actuals 202607 -- current 6800, prior was 6350

## last bonus was $38800 in account

annual_income = (b_monthly + a_monthly) * 12 
print(f"Annual projected net income: ${annual_income:,.2f}")

Annual projected net income: $273,329.52


# SUMMARY (NET POSITION)

In [108]:
# Example — replace with your own numbers
INTEREST_RATE = 6.09+.25+.25

# maximum interest rate is 7.9% whilst holding everything constant
monthly_mortgage = monthly_payment(2273090, INTEREST_RATE, 30)
INFLATION = 1.00  # Example inflation rate
print("SUMMARY TABLE")
print(f"Annual net income: ${annual_income:,.2f}")
expense = day_care_annual_cost + (monthly_mortgage * 12) + everyday_annual*INFLATION
print(f"Annual expense: ${expense:,.2f}")
print("---------------------------------")
print(f"Net position: ${annual_income - expense:,.2f}")
print("-------------------------------------")

print(f"DayCare annual cost: ${day_care_annual_cost:,.2f} -- {day_care_annual_cost/annual_income*100:,.2f}% -- ${Daily_rate}/day, {days_per_week} days/week")
print(f"Mortgage expense: ${monthly_mortgage * 12:,.2f} -- {monthly_mortgage*12/annual_income*100:,.2f}% -- {INTEREST_RATE}% interest rate, ${monthly_mortgage:,.2f} monthly repayment")
print(f"Everyday expenses: ${everyday_annual*INFLATION:,.2f} -- {everyday_annual*INFLATION/annual_income*100:,.2f}% -- ${everyday_annual*INFLATION/12:,.2f}/month on average, within inflation {(INFLATION-1)*100:,.1f}% ")



# $250K for a year


SUMMARY TABLE
Annual net income: $273,329.52
Annual expense: $248,173.11
---------------------------------
Net position: $25,156.41
-------------------------------------
DayCare annual cost: $35,360.00 -- 12.94% -- $170/day, 4 days/week
Mortgage expense: $174,343.17 -- 63.78% -- 6.59% interest rate, $14,528.60 monthly repayment
Everyday expenses: $38,469.94 -- 14.07% -- $3,205.83/month on average, within inflation 0.0% 
